In [1]:
import os
import sys

os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

print("Python:", sys.executable)
print("Python worker:", os.environ["PYSPARK_PYTHON"])

Python: C:\Users\Lenovo\anaconda3\python.exe
Python worker: C:\Users\Lenovo\anaconda3\python.exe


In [24]:
import os
import sys

os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable
os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"

from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .master("local[2]")
    .appName("PySpark_Hackathon")
    .config("spark.driver.host", "127.0.0.1")
    .config("spark.driver.bindAddress", "127.0.0.1")
    .config("spark.pyspark.python", sys.executable)
    .config("spark.pyspark.driver.python", sys.executable)
    .getOrCreate()
)

sc = spark.sparkContext

print("Master:", sc.master)
print("Driver host:", sc.getConf().get("spark.driver.host"))
print("Bind address:", sc.getConf().get("spark.driver.bindAddress"))
print("Python:", sys.executable)

Master: local[*]
Driver host: 127.0.0.1
Bind address: 127.0.0.1
Python: C:\Users\Lenovo\anaconda3\python.exe


In [27]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, trim, upper, when

data = [
    (1, " Aman ", "CSE", "85"),
    (2, "Priya", "IT", "92"),
    (3, None, "CSE", "67"),
    (4, "Rahul", "IT", None),
    (5, " Aman ", "CSE", "85"),
    (6, "Neha", "ECE", "105"),
    (7, "Ravi", None, "55"),
    (8, "Kiran", "cse", "78")
]

columns = ["student_id", "name", "department", "marks"]

dirty_df = spark.createDataFrame(data, columns)

dirty_df.show()
dirty_df.printSchema()

+----------+------+----------+-----+
|student_id|  name|department|marks|
+----------+------+----------+-----+
|         1| Aman |       CSE|   85|
|         2| Priya|        IT|   92|
|         3|  NULL|       CSE|   67|
|         4| Rahul|        IT| NULL|
|         5| Aman |       CSE|   85|
|         6|  Neha|       ECE|  105|
|         7|  Ravi|      NULL|   55|
|         8| Kiran|       cse|   78|
+----------+------+----------+-----+

root
 |-- student_id: long (nullable = true)
 |-- name: string (nullable = true)
 |-- department: string (nullable = true)
 |-- marks: string (nullable = true)



In [6]:

dirty_df.filter(col("name").isNull()).show()

+----------+----+----------+-----+
|student_id|name|department|marks|
+----------+----+----------+-----+
|         3|NULL|       CSE|   67|
+----------+----+----------+-----+



In [26]:
dirty_df.show()

+----------+------+----------+-----+
|student_id|  name|department|marks|
+----------+------+----------+-----+
|         1| Aman |       CSE|   85|
|         2| Priya|        IT|   92|
|         3|  NULL|       CSE|   67|
|         4| Rahul|        IT| NULL|
|         5| Aman |       CSE|   85|
|         6|  Neha|       ECE|  105|
|         7|  Ravi|      NULL|   55|
|         8| Kiran|       cse|   78|
+----------+------+----------+-----+



In [8]:
dirty_df.filter(col("department").isNull()).show()

+----------+----+----------+-----+
|student_id|name|department|marks|
+----------+----+----------+-----+
|         7|Ravi|      NULL|   55|
+----------+----+----------+-----+



In [9]:
from pyspark.sql.functions import col
dirty_df.filter(col("department").isNull()).show()

+----------+----+----------+-----+
|student_id|name|department|marks|
+----------+----+----------+-----+
|         7|Ravi|      NULL|   55|
+----------+----+----------+-----+



In [10]:
dirty_df.filter(col("marks").isNull()).show()

+----------+-----+----------+-----+
|student_id| name|department|marks|
+----------+-----+----------+-----+
|         4|Rahul|        IT| NULL|
+----------+-----+----------+-----+



In [11]:
dirty_df.dropna(
    subset=["name","department","marks"],
    how= "all"
).show()

+----------+------+----------+-----+
|student_id|  name|department|marks|
+----------+------+----------+-----+
|         1| Aman |       CSE|   85|
|         2| Priya|        IT|   92|
|         3|  NULL|       CSE|   67|
|         4| Rahul|        IT| NULL|
|         5| Aman |       CSE|   85|
|         6|  Neha|       ECE|  105|
|         7|  Ravi|      NULL|   55|
|         8| Kiran|       cse|   78|
+----------+------+----------+-----+



In [43]:
df = dirty_df.fillna({
    "name":"Unknown",
    "department":"Not assigned",
    "marks":"0"
})

df.show()

+----------+-------+------------+-----+
|student_id|   name|  department|marks|
+----------+-------+------------+-----+
|         1|  Aman |         CSE|   85|
|         2|  Priya|          IT|   92|
|         3|Unknown|         CSE|   67|
|         4|  Rahul|          IT|    0|
|         5|  Aman |         CSE|   85|
|         6|   Neha|         ECE|  105|
|         7|   Ravi|Not assigned|   55|
|         8|  Kiran|         cse|   78|
+----------+-------+------------+-----+



In [13]:
df = df.dropDuplicates(
    ["name","department","marks"]
)
df.show()

+----------+-------+------------+-----+
|student_id|   name|  department|marks|
+----------+-------+------------+-----+
|         1|  Aman |         CSE|   85|
|         2|  Priya|          IT|   92|
|         3|Unknown|         CSE|   67|
|         4|  Rahul|          IT|    0|
|         6|   Neha|         ECE|  105|
|         7|   Ravi|Not assigned|   55|
|         8|  Kiran|         cse|   78|
+----------+-------+------------+-----+



In [14]:
df = df.withColumn("marks",col("marks").cast("int"))
df.printSchema()
df.show()

root
 |-- student_id: long (nullable = true)
 |-- name: string (nullable = false)
 |-- department: string (nullable = false)
 |-- marks: integer (nullable = true)

+----------+-------+------------+-----+
|student_id|   name|  department|marks|
+----------+-------+------------+-----+
|         1|  Aman |         CSE|   85|
|         2|  Priya|          IT|   92|
|         3|Unknown|         CSE|   67|
|         4|  Rahul|          IT|    0|
|         6|   Neha|         ECE|  105|
|         7|   Ravi|Not assigned|   55|
|         8|  Kiran|         cse|   78|
+----------+-------+------------+-----+



In [15]:
df.filter(col("marks").isNull()).show()

+----------+----+----------+-----+
|student_id|name|department|marks|
+----------+----+----------+-----+
+----------+----+----------+-----+



In [16]:
df = (
    df
    .withColumn("name",trim(col("name")))
    .withColumn("department",upper(trim(col("department"))))
)

df.show()

+----------+-------+------------+-----+
|student_id|   name|  department|marks|
+----------+-------+------------+-----+
|         1|   Aman|         CSE|   85|
|         2|  Priya|          IT|   92|
|         3|Unknown|         CSE|   67|
|         4|  Rahul|          IT|    0|
|         6|   Neha|         ECE|  105|
|         7|   Ravi|NOT ASSIGNED|   55|
|         8|  Kiran|         CSE|   78|
+----------+-------+------------+-----+



In [17]:
df.show()

+----------+-------+------------+-----+
|student_id|   name|  department|marks|
+----------+-------+------------+-----+
|         1|   Aman|         CSE|   85|
|         2|  Priya|          IT|   92|
|         3|Unknown|         CSE|   67|
|         4|  Rahul|          IT|    0|
|         6|   Neha|         ECE|  105|
|         7|   Ravi|NOT ASSIGNED|   55|
|         8|  Kiran|         CSE|   78|
+----------+-------+------------+-----+



In [30]:
df.printSchema()

root
 |-- student_id: long (nullable = true)
 |-- name: string (nullable = true)
 |-- department: string (nullable = true)
 |-- marks: string (nullable = true)



In [37]:
df = df.withColumn("marks",col("marks").cast("int"))
df.printSchema()
df.show()

root
 |-- student_id: long (nullable = true)
 |-- name: string (nullable = true)
 |-- department: string (nullable = true)
 |-- marks: integer (nullable = true)

+----------+------+----------+-----+
|student_id|  name|department|marks|
+----------+------+----------+-----+
|         1| Aman |       CSE|   85|
|         2| Priya|        IT|   92|
|         3|  NULL|       CSE|   67|
|         4| Rahul|        IT|   78|
+----------+------+----------+-----+



In [39]:
invalid_marks = df.filter(
    (col("marks")<0) | (col("marks")>100)
)
invalid_marks.collect()

[]

In [40]:
df.select("marks").show(20)

+-----+
|marks|
+-----+
|   85|
|   92|
|   67|
|   78|
+-----+



In [41]:
df.show()

+----------+------+----------+-----+
|student_id|  name|department|marks|
+----------+------+----------+-----+
|         1| Aman |       CSE|   85|
|         2| Priya|        IT|   92|
|         3|  NULL|       CSE|   67|
|         4| Rahul|        IT|   78|
+----------+------+----------+-----+



In [44]:
df = df.withColumn(
    "name",
    trim(col("name"))
)
df.show()

+----------+-------+------------+-----+
|student_id|   name|  department|marks|
+----------+-------+------------+-----+
|         1|   Aman|         CSE|   85|
|         2|  Priya|          IT|   92|
|         3|Unknown|         CSE|   67|
|         4|  Rahul|          IT|    0|
|         5|   Aman|         CSE|   85|
|         6|   Neha|         ECE|  105|
|         7|   Ravi|Not assigned|   55|
|         8|  Kiran|         cse|   78|
+----------+-------+------------+-----+



In [45]:
df = df.withColumn(
    "department",
    upper(trim(col("department")))
)

df.show()

+----------+-------+------------+-----+
|student_id|   name|  department|marks|
+----------+-------+------------+-----+
|         1|   Aman|         CSE|   85|
|         2|  Priya|          IT|   92|
|         3|Unknown|         CSE|   67|
|         4|  Rahul|          IT|    0|
|         5|   Aman|         CSE|   85|
|         6|   Neha|         ECE|  105|
|         7|   Ravi|NOT ASSIGNED|   55|
|         8|  Kiran|         CSE|   78|
+----------+-------+------------+-----+



In [47]:
df = df.withColumn(
    "data_status",
    when(col("name").isNull(),"Missing Name")
    .when(col("department").isNull(),"Missing Department")
    .when(col("marks").isNull(),"Missing Marks")
    .when((col("marks")<0) | (col("marks")>100),"Invalid Marks")
    .otherwise("Valid")
    
)


In [48]:
df.show()

+----------+-------+------------+-----+-------------+
|student_id|   name|  department|marks|  data_status|
+----------+-------+------------+-----+-------------+
|         1|   Aman|         CSE|   85|        Valid|
|         2|  Priya|          IT|   92|        Valid|
|         3|Unknown|         CSE|   67|        Valid|
|         4|  Rahul|          IT|    0|        Valid|
|         5|   Aman|         CSE|   85|        Valid|
|         6|   Neha|         ECE|  105|Invalid Marks|
|         7|   Ravi|NOT ASSIGNED|   55|        Valid|
|         8|  Kiran|         CSE|   78|        Valid|
+----------+-------+------------+-----+-------------+



In [49]:
df.printSchema()

root
 |-- student_id: long (nullable = true)
 |-- name: string (nullable = false)
 |-- department: string (nullable = false)
 |-- marks: string (nullable = false)
 |-- data_status: string (nullable = false)



In [50]:
from pyspark.sql.functions import (
    col, count, sum, avg, min, max
)

data = [
    (1, "Aman", "CSE", 85),
    (2, "Priya", "IT", 92),
    (3, "Rahul", "CSE", 67),
    (4, "Neha", "IT", 88),
    (5, "Ravi", "ECE", 76),
    (6, "Kiran", "CSE", 95),
    (7, "Simran", "ECE", 81)
]

columns = ["student_id", "name", "department", "marks"]

students_df = spark.createDataFrame(data, columns)

students_df.show()

+----------+------+----------+-----+
|student_id|  name|department|marks|
+----------+------+----------+-----+
|         1|  Aman|       CSE|   85|
|         2| Priya|        IT|   92|
|         3| Rahul|       CSE|   67|
|         4|  Neha|        IT|   88|
|         5|  Ravi|       ECE|   76|
|         6| Kiran|       CSE|   95|
|         7|Simran|       ECE|   81|
+----------+------+----------+-----+



In [53]:
students_df.groupBy("department").count().show()

+----------+-----+
|department|count|
+----------+-----+
|       CSE|    3|
|        IT|    2|
|       ECE|    2|
+----------+-----+



In [54]:
pip install pyspark

Note: you may need to restart the kernel to use updated packages.


In [55]:
students_df.groupBy("department").count().show()

+----------+-----+
|department|count|
+----------+-----+
|       CSE|    3|
|        IT|    2|
|       ECE|    2|
+----------+-----+



In [60]:
students_df.groupBy("department").agg(
    avg("marks").alias("Average_marks")
).show()

+----------+-----------------+
|department|    Average_marks|
+----------+-----------------+
|       CSE|82.33333333333333|
|        IT|             90.0|
|       ECE|             78.5|
+----------+-----------------+



In [62]:
students_df.groupBy("department").agg(
    min("marks").alias("Minimum marks"),
    max("marks").alias("Maximum marks")
).show()

+----------+-------------+-------------+
|department|Minimum marks|Maximum marks|
+----------+-------------+-------------+
|       CSE|           67|           95|
|        IT|           88|           92|
|       ECE|           76|           81|
+----------+-------------+-------------+



In [64]:
students_df.groupBy("department").sum("marks").show()

+----------+----------+
|department|sum(marks)|
+----------+----------+
|       CSE|       247|
|        IT|       180|
|       ECE|       157|
+----------+----------+

